In [ ]:
import numpy as np
import torch
import torch.nn as nn
import wandb
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

In [ ]:
harm_X     = np.load("/content/drive/MyDrive/128x128/harmonics_lid_driven_cavity_X.npz")['data'][:500]
harm_Y     = np.load("/content/drive/MyDrive/128x128/harmonics_lid_driven_cavity_Y.npz")['data'][:500]

nurbs_X    = np.load("/content/drive/MyDrive/128x128/nurbs_lid_driven_cavity_X.npz")["data"][:500]
nurbs_Y    = np.load("/content/drive/MyDrive/128x128/nurbs_lid_driven_cavity_Y.npz")["data"][:500]

skeleton_X = np.load("/content/drive/MyDrive/128x128/skelneton_lid_driven_cavity_X.npz")["data"][:500]
skeleton_Y = np.load("/content/drive/MyDrive/128x128/skelneton_lid_driven_cavity_Y.npz")["data"][:500]

print("Harmonics X:", harm_X.shape)
print("NURBS X:",     nurbs_X.shape)
print("Skeleton X:",  skeleton_X.shape)

Harmonics X: (500, 3, 128, 128)
NURBS X: (500, 3, 128, 128)
Skeleton X: (500, 3, 128, 128)


In [ ]:
def downsample(arr, size=64):
    t = torch.tensor(arr).float()
    return F.interpolate(t, size=(size, size), mode='bilinear', align_corners=False).numpy()

harm_X     = downsample(harm_X)
harm_Y     = downsample(harm_Y)
nurbs_X    = downsample(nurbs_X)
nurbs_Y    = downsample(nurbs_Y)
skeleton_X = downsample(skeleton_X)
skeleton_Y = downsample(skeleton_Y)

print("After downsample:", harm_X.shape)  # (500, 3, 64, 64)

After downsample: (500, 3, 64, 64)


 # Combine and keep u, v, p

In [ ]:
X = np.concatenate([harm_X, nurbs_X, skeleton_X], axis=0)
Y = np.concatenate([harm_Y, nurbs_Y, skeleton_Y], axis=0)

Y = Y[:, :3]  # keep only u, v, p

print("X:", X.shape)  # (1500, 3, 64, 64)
print("Y:", Y.shape)  # (1500, 3, 64, 64)

X: (1500, 3, 64, 64)
Y: (1500, 3, 64, 64)


# Convert to tensors and normalize

In [ ]:
X = torch.tensor(X).float()
Y = torch.tensor(Y).float()

X_mean = X.mean(dim=(0,2,3), keepdim=True)
X_std  = X.std(dim=(0,2,3),  keepdim=True) + 1e-6
X = (X - X_mean) / X_std

Y_mean = Y.mean(dim=(0,2,3), keepdim=True)
Y_std  = Y.std(dim=(0,2,3),  keepdim=True) + 1e-6
Y = (Y - Y_mean) / Y_std

# Train and Test split

In [ ]:
train_size = int(0.8 * X.shape[0])

X_train = X[:train_size]
Y_train = Y[:train_size]

X_test  = X[train_size:]
Y_test  = Y[train_size:]

print("Train:", X_train.shape)
print("Test:",  X_test.shape)

Train: torch.Size([1200, 3, 64, 64])
Test: torch.Size([300, 3, 64, 64])


# DeepONet
DeepONet has two inputs — a branch net that encodes the input function, and a trunk net that encodes the query coordinates where you want the output evaluated.

In [ ]:
# Branch input: flatten the spatial input field
# (B, 3, 64, 64) -> (B, 3*64*64)
X_train_branch = X_train.reshape(X_train.shape[0], -1)
X_test_branch  = X_test.reshape(X_test.shape[0],  -1)

# Trunk input: query coordinates (same for every sample)
# Build a normalized grid of (x, y) points -> shape (64*64, 2)
gridx = torch.linspace(0, 1, 64)
gridy = torch.linspace(0, 1, 64)
grid_y, grid_x = torch.meshgrid(gridy, gridx, indexing='ij')
trunk_input = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)  # (4096, 2)

# Output: flatten spatial output
# (B, 3, 64, 64) -> (B, 3, 4096) -> (B, 4096, 3)
Y_train_flat = Y_train.reshape(Y_train.shape[0], 3, -1).permute(0, 2, 1)
Y_test_flat  = Y_test.reshape(Y_test.shape[0],  3, -1).permute(0, 2, 1)

print("Branch train:", X_train_branch.shape)
print("Trunk:        ", trunk_input.shape)
print("Output train: ", Y_train_flat.shape)

Branch train: torch.Size([1200, 12288])
Trunk:         torch.Size([4096, 2])
Output train:  torch.Size([1200, 4096, 3])


In [ ]:
class BranchNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim)
        )

    def forward(self, x):
        return self.net(x)  # (B, latent_dim)


class TrunkNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim)
        )

    def forward(self, x):
        return self.net(x)  # (N_points, latent_dim)


class DeepONet(nn.Module):
    def __init__(self, branch_input, trunk_input=2, hidden_dim=128, latent_dim=128, out_channels=3):
        super().__init__()
        self.branch = BranchNet(branch_input, hidden_dim, latent_dim * out_channels)
        self.trunk  = TrunkNet(trunk_input,  hidden_dim, latent_dim * out_channels)
        self.bias   = nn.Parameter(torch.zeros(out_channels))
        self.latent_dim   = latent_dim
        self.out_channels = out_channels

    def forward(self, branch_x, trunk_x):
        # branch_x: (B, branch_input)
        # trunk_x:  (N, 2)

        b = self.branch(branch_x)  # (B, latent_dim * out_channels)
        t = self.trunk(trunk_x)    # (N, latent_dim * out_channels)

        # Reshape for per-channel dot product
        B = b.shape[0]
        N = t.shape[0]

        b = b.reshape(B, self.out_channels, self.latent_dim)  # (B, 3, latent_dim)
        t = t.reshape(N, self.out_channels, self.latent_dim)  # (N, 3, latent_dim)

        # Dot product: (B, 3, latent) x (N, 3, latent) -> (B, N, 3)
        out = torch.einsum('bci,nci->bnc', b, t)  # (B, N, 3)
        out = out + self.bias                      # add learnable bias per channel

        return out  # (B, 4096, 3)

In [ ]:
def relative_l2(pred, target):
    diff_norm   = torch.norm(pred.reshape(pred.shape[0], -1) - target.reshape(target.shape[0], -1), dim=1)
    target_norm = torch.norm(target.reshape(target.shape[0], -1), dim=1)
    return (diff_norm / target_norm).mean()

In [ ]:
import wandb
import os
os.environ['WANDB_API_KEY']= "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"

wandb.init(
    project="ppt-deeponet",
    name="run2-deeponet-64x64",
    config={
        "epochs":     200,
        "batch_size": 32,
        "lr":         1e-3,
        "hidden_dim": 128,
        "latent_dim": 128,
        "grid_size":  64,
        "samples":    1500,
    }
)

In [ ]:
epochs     = 200
batch_size = 32
lr         = 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"

model     = DeepONet(branch_input=3*64*64, hidden_dim=128, latent_dim=128, out_channels=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn   = nn.MSELoss()

trunk_input_dev = trunk_input.to(device)  # fixed, move once

train_loader = DataLoader(TensorDataset(X_train_branch, Y_train_flat), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_branch,  Y_test_flat),  batch_size=batch_size, shuffle=False)

for epoch in range(epochs):

    # --- TRAINING ---
    model.train()
    train_loss_epoch = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb, trunk_input_dev)   # (B, 4096, 3)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_epoch += loss.item()

    train_loss_epoch /= len(train_loader)

    # --- TESTING ---
    model.eval()
    test_loss_epoch = 0.0
    rel_l2_epoch    = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb, trunk_input_dev)
            test_loss_epoch += loss_fn(pred, yb).item()
            rel_l2_epoch    += relative_l2(pred, yb).item()

    test_loss_epoch /= len(test_loader)
    rel_l2_epoch    /= len(test_loader)

    # --- LOGGING ---
    wandb.log({
        "epoch":             epoch,
        "train_loss":        train_loss_epoch,
        "test_loss":         test_loss_epoch,
        "relative_l2_error": rel_l2_epoch,
    })

    print(f"Epoch {epoch:03d}/{epochs} | Train: {train_loss_epoch:.6f} | Test: {test_loss_epoch:.6f} | Rel L2: {rel_l2_epoch:.6f}")

wandb.finish()

Epoch 000/200 | Train: 1.085531 | Test: 0.553581 | Rel L2: 0.754350
Epoch 001/200 | Train: 0.521786 | Test: 0.507634 | Rel L2: 0.741410
Epoch 002/200 | Train: 0.461473 | Test: 0.475350 | Rel L2: 0.722151
Epoch 003/200 | Train: 0.436624 | Test: 0.922384 | Rel L2: 0.936664
Epoch 004/200 | Train: 0.490920 | Test: 0.459022 | Rel L2: 0.709666
Epoch 005/200 | Train: 0.425893 | Test: 0.461406 | Rel L2: 0.711779
Epoch 006/200 | Train: 0.436282 | Test: 0.440909 | Rel L2: 0.697202
Epoch 007/200 | Train: 0.403962 | Test: 0.544488 | Rel L2: 0.763053
Epoch 008/200 | Train: 0.423466 | Test: 0.430360 | Rel L2: 0.674722
Epoch 009/200 | Train: 0.394841 | Test: 0.410713 | Rel L2: 0.669109
Epoch 010/200 | Train: 0.382398 | Test: 0.405063 | Rel L2: 0.662624
Epoch 011/200 | Train: 0.362416 | Test: 0.377495 | Rel L2: 0.637343
Epoch 012/200 | Train: 0.354754 | Test: 0.394838 | Rel L2: 0.649054
Epoch 013/200 | Train: 0.350752 | Test: 0.408916 | Rel L2: 0.663816
Epoch 014/200 | Train: 0.339532 | Test: 0.386101

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
relative_l2_error,█▇▅▄▄▃▄▃▃▄▃▃▂▃▃▂▂▂▁▂▂▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
test_loss,███▅▅▃▃▅▅▃▄▅▇▃▄▃▃▂▃▃▂▄▂▃▃▁▂▂▂▄▃▂▁▄▄▃▃▂▃▃
train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
relative_l2_error,0.43103
test_loss,0.29488
train_loss,0.1097
